# Triton Fused Add + RMSNorm: Colab Master Notebook

This notebook runs the full project end-to-end on Google Colab GPU and extracts resume-ready numbers.

What you get:
- Correctness validation
- Benchmark latency and throughput for fp16 and bf16
- Speedup vs eager PyTorch and vs torch.compile
- Percent of peak memory bandwidth
- Auto-generated resume bullet points

Expected runtime on Colab T4: around 15 to 30 minutes depending on runtime load.

## Cell 1: Select GPU Runtime
In Colab, go to Runtime -> Change runtime type -> Hardware accelerator -> GPU. Then run next cells.

In [ ]:
# Environment and GPU check
import os
import json
import subprocess
from pathlib import Path

def run(cmd, check=True):
    print(f"\n$ {cmd}")
    proc = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        if proc.stderr:
            print(proc.stderr)
        if check:
            raise RuntimeError(f"Command failed with code {proc.returncode}: {cmd}")
    return proc

run('python --version')
run('pip --version')
run('nvidia-smi')

In [ ]:
# Set your GitHub repo URL here
REPO_URL = 'https://github.com/your-username/triton-fused-add-rmsnorm.git'
REPO_DIR = '/content/triton-fused-add-rmsnorm'

if os.path.exists(REPO_DIR):
    run(f'rm -rf {REPO_DIR}')
run(f'git clone {REPO_URL} {REPO_DIR}')
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
run('ls -la')

In [ ]:
# Install dependencies
run('pip install -U pip')
run('pip install -r requirements.txt')
run('python - << PY\nimport torch, triton\nprint("torch:", torch.__version__)\nprint("triton:", triton.__version__)\nprint("cuda available:", torch.cuda.is_available())\nif torch.cuda.is_available():\n    print("gpu:", torch.cuda.get_device_name(0))\nPY')

In [ ]:
# Run correctness tests
run('pytest tests -q')

## Benchmarks
The next cells run benchmark matrices and write JSON + Markdown outputs.

In [ ]:
# fp16 benchmark (includes eager, torch.compile, triton)
run('python bench.py --dtype fp16 --out results_fp16')

In [ ]:
# bf16 benchmark (includes eager, torch.compile, triton)
run('python bench.py --dtype bf16 --out results_bf16')

In [ ]:
# Optional: benchmark without torch.compile baseline (useful if compile is unstable in your runtime)
# run('python bench.py --dtype fp16 --no-compile --out results_fp16_nocompile')
# run('python bench.py --dtype bf16 --no-compile --out results_bf16_nocompile')

In [ ]:
# Parse benchmark files and compute aggregate metrics for resume
import statistics

def load_results(path):
    with open(path, 'r') as f:
        return json.load(f)

def summarize(tag, payload):
    rows = payload['rows']
    meta = payload['meta']

    speedup_vs_eager = [r['speedup_vs_eager'] for r in rows]
    triton_pct_peak = [r['triton_pct_peak'] for r in rows if 'triton_pct_peak' in r]
    triton_gbps = [r['triton_gbps'] for r in rows]

    summary = {
        'dtype': tag,
        'gpu': meta.get('gpu'),
        'peak_gbps': meta.get('peak_gbps'),
        'n_shapes': len(rows),
        'speedup_vs_eager_min': min(speedup_vs_eager),
        'speedup_vs_eager_median': statistics.median(speedup_vs_eager),
        'speedup_vs_eager_max': max(speedup_vs_eager),
        'triton_gbps_min': min(triton_gbps),
        'triton_gbps_median': statistics.median(triton_gbps),
        'triton_gbps_max': max(triton_gbps),
    }

    if triton_pct_peak:
        summary.update({
            'pct_peak_min': min(triton_pct_peak),
            'pct_peak_median': statistics.median(triton_pct_peak),
            'pct_peak_max': max(triton_pct_peak),
        })

    if 'speedup_vs_compile' in rows[0]:
        speedup_vs_compile = [r['speedup_vs_compile'] for r in rows]
        summary.update({
            'speedup_vs_compile_min': min(speedup_vs_compile),
            'speedup_vs_compile_median': statistics.median(speedup_vs_compile),
            'speedup_vs_compile_max': max(speedup_vs_compile),
        })

    best_shape = max(rows, key=lambda r: r['speedup_vs_eager'])
    summary['best_shape_vs_eager'] = {
        'tokens': best_shape['tokens'],
        'hidden': best_shape['hidden'],
        'speedup': best_shape['speedup_vs_eager'],
    }

    return summary

fp16_payload = load_results('results_fp16.json')
bf16_payload = load_results('results_bf16.json')

fp16_summary = summarize('fp16', fp16_payload)
bf16_summary = summarize('bf16', bf16_payload)

print('FP16 summary:')
print(json.dumps(fp16_summary, indent=2))
print('\nBF16 summary:')
print(json.dumps(bf16_summary, indent=2))

with open('resume_metrics_summary.json', 'w') as f:
    json.dump({'fp16': fp16_summary, 'bf16': bf16_summary}, f, indent=2)
print('\nWrote resume_metrics_summary.json')

In [ ]:
# Generate resume bullet candidates from measured values
def bullet_lines(s):
    lines = []
    lines.append(
        f"Built a Triton fused residual-add + RMSNorm kernel and benchmarked across {s['n_shapes']} shapes on {s['gpu']} ({s['dtype']}); achieved {s['speedup_vs_eager_median']:.2f}x median and {s['speedup_vs_eager_max']:.2f}x peak speedup vs eager PyTorch."
    )
    if 'speedup_vs_compile_median' in s:
        lines.append(
            f"Compared against torch.compile baseline: {s['speedup_vs_compile_median']:.2f}x median and {s['speedup_vs_compile_max']:.2f}x best-case speedup."
        )
    if 'pct_peak_median' in s:
        lines.append(
            f"Reached {s['triton_gbps_median']:.0f} GB/s median effective bandwidth ({s['pct_peak_median']:.0f}% of peak), peaking at {s['triton_gbps_max']:.0f} GB/s ({s['pct_peak_max']:.0f}% of peak)."
        )
    lines.append(
        f"Best speedup observed at tokens={s['best_shape_vs_eager']['tokens']}, hidden={s['best_shape_vs_eager']['hidden']} with {s['best_shape_vs_eager']['speedup']:.2f}x vs eager."
    )
    return lines

for label, summ in [('FP16', fp16_summary), ('BF16', bf16_summary)]:
    print(f'\n=== {label} Resume Bullets ===')
    for line in bullet_lines(summ):
        print('- ' + line)

## Optional: Nsight Compute
If your environment has ncu installed, run the profiling script for memory throughput details.

In [ ]:
# Optional Nsight Compute profiling (may not be available on free Colab)
run('which ncu', check=False)
# run('bash scripts/profile_ncu.sh')

## Output files
After successful run, key artifacts are:
- results_fp16.json and results_fp16.md
- results_bf16.json and results_bf16.md
- resume_metrics_summary.json

Download these from Colab file browser and keep them with your project write-up.